In [12]:
import os
import pandas as pd
import json
import anthropic
from dotenv import load_dotenv

os.chdir(r"F:\finance-ai-course")
load_dotenv()

# Use absolute path — works no matter what the working directory is
DATA_PATH = r"F:\finance-ai-course\data\trial_balance.csv"

def get_account_balance(account_code: str, period: str, entity: str) -> dict:
    df = pd.read_csv(DATA_PATH)
    row = df[(df["account_code"].astype(str) == account_code) &
             (df["period"] == period) &
             (df["entity"] == entity)]
    if row.empty:
        return {"error": f"No data for {account_code} / {period} / {entity}"}
    r = row.iloc[0]
    return {"account_code": account_code, "account_name": r["account_name"],
            "debit": float(r["debit"]), "credit": float(r["credit"]),
            "net": float(r["debit"]) - float(r["credit"])}

def check_trial_balance(period: str, entity: str) -> dict:
    df = pd.read_csv(DATA_PATH)
    filtered = df[(df["period"] == period) & (df["entity"] == entity)]
    total_dr = filtered["debit"].sum()
    total_cr = filtered["credit"].sum()
    diff = total_dr - total_cr
    return {"period": period, "entity": entity,
            "total_debits": total_dr, "total_credits": total_cr,
            "difference": diff, "balanced": abs(diff) < 0.01}

def flag_round_numbers(entity: str, period: str, threshold: int = 10000) -> list:
    df = pd.read_csv(DATA_PATH)
    filtered = df[(df["entity"] == entity) & (df["period"] == period)]
    flags = []
    for _, row in filtered.iterrows():
        for col in ["debit", "credit"]:
            val = float(row[col])
            if val >= threshold and val % 1000 == 0:
                flags.append({"account": row["account_name"], "amount": val, "side": col})
    return flags

print("Tools loaded ✓")
print("Data file exists:", os.path.exists(DATA_PATH))

Tools loaded ✓
Data file exists: True


In [13]:
TOOL_SCHEMAS = [
    {"name": "get_account_balance",
     "description": "Get the debit/credit balance for a specific account, period and entity.",
     "input_schema": {"type": "object",
         "properties": {
             "account_code": {"type": "string"},
             "period": {"type": "string"},
             "entity": {"type": "string"}},
         "required": ["account_code", "period", "entity"]}},
    {"name": "check_trial_balance",
     "description": "Check if debits equal credits for a given period and entity.",
     "input_schema": {"type": "object",
         "properties": {"period": {"type": "string"}, "entity": {"type": "string"}},
         "required": ["period", "entity"]}},
    {"name": "flag_round_numbers",
     "description": "Flag suspiciously round transaction amounts.",
     "input_schema": {"type": "object",
         "properties": {"entity": {"type": "string"}, "period": {"type": "string"},
                        "threshold": {"type": "integer"}},
         "required": ["entity", "period"]}},
]

TOOL_MAP = {
    "get_account_balance": get_account_balance,
    "check_trial_balance": check_trial_balance,
    "flag_round_numbers": flag_round_numbers,
}

print("Tool schemas ready ✓")

Tool schemas ready ✓


In [17]:
client = anthropic.Anthropic()

def run_finance_agent(task: str) -> str:
    messages = [{"role": "user", "content": task}]
    
    while True:
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=2048,
            system="""You are a GL audit agent. Use your tools to analyse 
financial data and produce a structured audit report. Always check the 
trial balance first, then investigate specific accounts, then flag anomalies.""",
            tools=TOOL_SCHEMAS,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            return response.content[0].text
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  → Agent calling: {block.name}({block.input})")
                fn = TOOL_MAP[block.name]
                result = fn(**block.input)
                # Convert everything to JSON-safe types
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result, default=str)
                })
        
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

print("Agent ready ✓")

Agent ready ✓


In [18]:
task = """Perform a GL audit for EntityA, period 2025-12. 
1. Verify the trial balance is balanced
2. Check revenue accounts 4001 and 4002
3. Flag any round-number anomalies
4. Write a 3-point audit summary with risk rating Low/Medium/High"""

print("Running GL Audit Agent...\n")
report = run_finance_agent(task)
print("\n=== AUDIT REPORT ===")
print(report)

Running GL Audit Agent...

  → Agent calling: check_trial_balance({'entity': 'EntityA', 'period': '2025-12'})
  → Agent calling: get_account_balance({'account_code': '4001', 'entity': 'EntityA', 'period': '2025-12'})
  → Agent calling: get_account_balance({'account_code': '4002', 'entity': 'EntityA', 'period': '2025-12'})
  → Agent calling: flag_round_numbers({'entity': 'EntityA', 'period': '2025-12'})

=== AUDIT REPORT ===
---

# 🔍 GL AUDIT REPORT

**Entity:** EntityA  
**Period:** 2025-12  
**Audit Date:** Generated by GL Audit Agent

---

## 1. Trial Balance Verification

| Metric | Value |
|--------|-------|
| Total Debits | $900,000.00 |
| Total Credits | $815,700.00 |
| **Difference** | **$84,300.00** |
| **Status** | ❌ **UNBALANCED** |

⚠️ **CRITICAL FINDING:** The trial balance does NOT balance. There is an $84,300 discrepancy between debits and credits. This requires immediate investigation and correction.

---

## 2. Revenue Account Analysis

| Account | Name | Debit | Credit

In [21]:
os.makedirs("day3", exist_ok=True)
with open("day3/gl_audit_agent.py", "w", encoding="utf-8") as f:
    f.write(script.replace("→", "->"))

print("Saved to day3/gl_audit_agent.py ✓")

Saved to day3/gl_audit_agent.py ✓
